In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
CREATE OR REPLACE VIEW market_view AS
SELECT *
FROM read_parquet('/app/data/**/*.snappy.parquet');
""")


In [6]:
df = con.sql("""
    SELECT distinct product_id, count(product_id) as row_count
    FROM market_view
    WHERE year = 2026 AND month = 6 AND day = 26 group by product_id order by count(product_id) desc
    """).fetchdf()
df

,product_id,row_count
0,BTC-USD,33785
1,ETH-USD,15214
2,SOL-USD,14621
3,XRP-USD,5611
4,AERO-USD,3209
5,DOGE-USD,1031
6,USDT-USD,764


In [7]:
vwap = con.sql("""
    SELECT product_id, SUM(price * last_size) as weighted_price, SUM(last_size) as cumulative_volume, SUM(price * last_size) / SUM(last_size) as vwap
    FROM market_view
    WHERE year = 2026 AND month = 6 AND day = 26 group by product_id
    """)
vwap

┌────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ product_id │   weighted_price   │ cumulative_volume  │        vwap         │
│  varchar   │       double       │       double       │       double        │
├────────────┼────────────────────┼────────────────────┼─────────────────────┤
│ SOL-USD    │  5902177.492623775 │  80627.01373327004 │   73.20347386484305 │
│ DOGE-USD   │ 194342.00017799996 │ 2568677.9000000004 │ 0.07565837669954646 │
│ XRP-USD    │ 4093100.7842980577 │ 3924066.4812169983 │   1.043076309713447 │
│ BTC-USD    │  36803061.38697763 │  615.3079303300026 │   59812.42817273519 │
│ AERO-USD   │ 277312.79481800005 │  575356.3000000003 │  0.4819844587049797 │
│ USDT-USD   │  751770.3689826998 │          752923.41 │  0.9984685812633981 │
│ ETH-USD    │ 10298334.965570262 │  6534.671632149989 │   1575.952939227029 │
└────────────┴────────────────────┴────────────────────┴─────────────────────┘